In [1]:
from fastmcp import FastMCP
from pathlib import Path
import logging
import warnings

# Suppress deprecation warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("fastmcp").setLevel(logging.WARNING)
mcp = FastMCP("HTTP file server")

In [3]:
BASE_DIR = Path.cwd() / "workspace"
BASE_DIR.mkdir(exist_ok=True)



In [4]:
def is_within_roots(path:Path) -> bool :
    """ Checks if path is within allowed root directory"""
    try:
        path.resolve().relative_to(BASE_DIR.resolve())
        return True
    except ValueError:
        return False

In [5]:
# adding read file tool
@mcp.tool
def read_file(filepath:str)-> str:
    """ Read a file from workspace directory"""
    path = BASE_DIR/ filepath
    if not is_within_roots(path):
        return f"Error: Access denied -path outside workpsace roots"
    if not path.exists():
        return f"Error: File not found :{filepath}"
    try:
        content = path.read_text()
        return content
    except Exception as e:
        return f" Error reading file:{str(e)}"

In [6]:
# add write file tool
@mcp.tool
def write_file(filepath: str, content:str) -> str:
    """Writes the content to a file in workspace directory """
    path = BASE_DIR/ filepath
    if not is_within_roots(path):
        return f"Error: Access denied -path outside workpsace roots"
    
    try:
        path.parent.mkdir(parents = True , exists_ok = True)
        path.write_text(content)
        return f"Successfully wrote {len(content)} characters to {filepath}"
    except Exception as e:
        return f"Error writing file: {str(e)}"    

In [7]:
@mcp.tool()
def list_files(directory: str = ".") -> str:
    """List files in a directory within the workspace."""
    path = BASE_DIR / directory

    if not is_within_roots(path):
        return f"Error: Access denied - path outside workspace roots"

    if not path.exists():
        return f"Error: Directory not found: {directory}"

    if not path.is_dir():
        return f"Error: Not a directory: {directory}"

    try:
        files = []
        for item in sorted(path.iterdir()):
            relative_path = item.relative_to(BASE_DIR)
            file_type = "DIR" if item.is_dir() else "FILE"
            size = item.stat().st_size if item.is_file() else 0
            files.append(f"{file_type}: {relative_path} ({size} bytes)")

        return "\n".join(files) if files else "Directory is empty"
    except Exception as e:
        return f"Error listing directory: {str(e)}"

In [8]:
@mcp.tool()
def analyze_code(code: str, focus: str = "quality") -> str:
    """Analyze code focusing on specified aspect.

    In a full MCP implementation with bidirectional communication,
    this tool would send a sampling/createMessage JSON-RPC request
    to the client. For this educational lab, we return a message
    indicating where sampling would occur.
    """
    return f"""[SAMPLING TRIGGER]
This tool would send a sampling/createMessage request to the client:

{{
  'method': 'sampling/createMessage',
  'params': {{
    'messages': [{{'role': 'user', 'content': {{
      'type': 'text',
      'text': 'Analyze this code for {focus}:\\n{code[:50]}...'
    }}}}}}],
    'maxTokens': 500
  }}
}}

The client would:
1. Show approval dialog to user
2. If approved, call LLM with the prompt
3. Return LLM response to server
4. Server would use response to complete analysis

Note: Full bidirectional sampling requires low-level MCP SDK.
This simplified version demonstrates the concept."""

In [9]:
@mcp.resource("file://workspace/{filename}")
def get_workspace_file(filename: str) -> str:
    """Read a file from the workspace as a resource."""
    path = BASE_DIR / filename
    if not is_within_roots(path):
        raise ValueError(f"Access denied - path outside workspace roots")
    if not path.exists():
        raise ValueError(f"File not found: {filename}")
    return path.read_text()

In [10]:
@mcp.prompt()
def review_code(filename: str) -> str:
    """Generate a prompt to review code from a file."""
    return f"""Please review the code in file '{filename}' and provide:
1. A summary of what the code does
2. Potential bugs or issues
3. Security concerns
4. Suggestions for improvements
5. Code quality assessment
Focus on readability, maintainability, and best practices."""
@mcp.prompt()
def analyze_security(filename: str) -> str:
    """Generate a prompt to analyze security of a file."""
    return f"""Perform a security analysis of '{filename}' focusing on:
1. Input validation and sanitization
2. Authentication and authorization checks
3. Potential injection vulnerabilities
4. Data exposure risks
5. Error handling security
Provide specific line numbers and remediation suggestions."""

In [ ]:
if __name__ == "__main__":
    print("Starting HTTP MCP Server on http://127.0.0.1:10000")
    print(f"Workspace roots: {BASE_DIR}")

    await mcp.run_async(
        transport="streamable-http",  # Specify HTTP transport
        host="127.0.0.1",
        port=10000
    )

Starting HTTP MCP Server on http://127.0.0.1:10000
Workspace roots: /Users/anirudh/Desktop/Courseera/IBM-RAG and agentic AI/Ibm-mcp/mcp_advanced_lab/workspace


╭────────────────────────────────────────────────────────────────────────────╮
│                                                                            │
│        _ __ ___  _____           __  __  _____________    ____    ____     │
│       _ __ ___ .'____/___ ______/ /_/  |/  / ____/ __ \  |___ \  / __ \    │
│      _ __ ___ / /_  / __ `/ ___/ __/ /|_/ / /   / /_/ /  ___/ / / / / /    │
│     _ __ ___ / __/ / /_/ (__  ) /_/ /  / / /___/ ____/  /  __/_/ /_/ /     │
│    _ __ ___ /_/    \____/____/\__/_/  /_/\____/_/      /_____(*)____/      │
│                                                                            │
│                                                                            │
│                                FastMCP  2.0                                │
│                                                                            │
│                                                                            │
│               🖥️ Server name:     HTTP file server                         │
│               📦 Transport:       Streamable-HTTP                          │
│               🔗 Server URL:      http://127.0.0.1:10000/mcp               │
│                                                                            │
│               🏎️ FastMCP version: 2.12.4                                   │
│               🤝 MCP SDK version: 1.15.0                                   │
│                                                                            │
│               📚 Docs:            https://gofastmcp.com                    │
│               🚀 Deploy:          https://fastmcp.cloud                    │
│                                                                            │
╰────────────────────────────────────────────────────────────────────────────╯

INFO:     Started server process [57604]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:10000 (Press CTRL+C to quit)
